In [1]:
import os
import sys

import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch import nn
from tqdm import tqdm

sys.path.append('/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis')
%cd '/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis/'

/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis


In [2]:
from torch.nn import BCEWithLogitsLoss

from architecture.classic_cbm import ClassicCBM, init_from_checkpoint
from cbm_datasets import Batch, get_dataloader, get_datasets
from utils.attribution_methods import AttributorBase, get_attribution_maps, get_attributor_by_name
from utils.loss import CELossFunnyBirds, EPGLoss

In [3]:
def create_summary_table(df: pd.DataFrame, filter_tag:list[str], reference_measure:str) -> pd.DataFrame:
    """Erstellt eine Tabelle mit den Bestwerten pro Run."""
    print("\nErstelle Summary Tabelle (Max-Werte)...")

    # filter df for each column where df[column] is true
    for tag in filter_tag:
        col = f"tag_{tag}"
        if col not in df.columns:
            raise ValueError(f"Tag-Spalte {col} existiert nicht im DataFrame.")
        df = df[df[col] == True]

    if df.empty:
        print("DataFrame nach Tag-Filter leer.")
        return df

    # Sicherstellen, dass innerhalb jedes Runs nach step sortiert ist
    df = df.sort_values(["run_id", "_step"]).copy()

    # Epoch als laufende Evaluation pro run_id (startet bei 0)
    df["epoch"] = df.groupby("run_id").cumcount()

    # Bestes F1-Concept pro Run auswählen
    best_per_run = (
        df.loc[df.groupby("run_id")[reference_measure].idxmax()]
        .reset_index(drop=True)
    )
    
    return best_per_run

In [4]:
DATA_PATH = 'thesis-figures/epg_cbm/results/1_epg_cbm_results.parquet'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = pd.read_parquet(DATA_PATH)

print("Using device:", device)

Using device: cuda


In [5]:
data.columns

Index(['_step', 'Loss EPG', 'F1 Concepts', 'F1 Labels', 'Runtime', 'Epoch',
       'run_id', 'Attribution Method', '$\lambda_\mathrm{EPG}$', 'EPG-level',
       'config_label', 'Dataset', 'tag_result', 'tag_baseline'],
      dtype='object')

In [6]:
from dataclasses import asdict, dataclass
from typing import Any


@dataclass
class EvalMetrics:
    # --- core eval metrics ---
    dice_mean: float
    iou_mean: float
    precision_concepts: float
    recall_concepts: float
    accuracy_concepts: float
    f1_labels: float
    accuracy_labels: float
    foreground_dice_scores: float
    concept_f1_micro: float
    concept_f1_macro: float
    avg_epg_loss: float

    # --- optional metadata (for dataframe concatenation later) ---
    concept_module: str | None = None
    segmentation_module: str | None = None
    dataset: str | None = None
    runtime: float | None = None

    # -------------------------
    # 2️⃣ Pretty renamed dict
    # -------------------------
    def to_pretty_dict(self) -> dict[str, Any]:
        return {
            "Mean Dice": self.dice_mean,
            "Mean IoU": self.iou_mean,
            r"Concept Activations $F_1$-Score (Macro)": self.concept_f1_macro,
            r"Concept Activations $F_1$-Score (Micro)": self.concept_f1_micro,
            "Precision Concepts": self.precision_concepts,
            "Recall Concepts": self.recall_concepts,
            "Concept Accuracy": self.accuracy_concepts,
            "EPG Loss": self.avg_epg_loss,
            r"Label $F_1$-Score": self.f1_labels,
            "Label Accuracy": self.accuracy_labels,
            "Foreground Dice": self.foreground_dice_scores,
            "Concept Module": self.concept_module,
            "Segmentation Module": self.segmentation_module,
            "Dataset": self.dataset,
            "Runtime": self.runtime,
        }

    # -------------------------
    # 3️⃣ DataFrame row
    # -------------------------
    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([self.to_pretty_dict()])

In [7]:
import torch


def evaluate(
    model: ClassicCBM,
    val_loader: torch.utils.data.DataLoader,
    device,
    concept_criterion: nn.Module,      # Hinzugefügt (war im Docstring, fehlte im Header)
    classification_criterion: nn.Module, # Hinzugefügt
    epg_criterion: nn.Module | None,
    attributor: AttributorBase | None,
):
    model.eval()

    total_samples = 0
    correctly_classified = 0
    
    # Listen für globale Metriken
    all_label_preds = []
    all_label_targets = []
    all_concept_preds = []
    all_concept_targets = []

    # Tracker für Verluste
    total_concept_loss = 0.0
    total_class_loss = 0.0
    total_epg_loss = 0.0

    # Tracker für Lokalisierung (Dice/IoU)
    total_dice = 0.0
    total_iou = 0.0
    
    # Konzept-Statistiken (Micro-Averaging)
    tp_concepts = 0
    fp_concepts = 0
    fn_concepts = 0
    tn_concepts = 0

    pbar = tqdm(val_loader, desc="[Eval]")

    with torch.no_grad(): # Standardmäßig für Eval, außer für Attributionen
        for batch in pbar:
            batch:Batch
            batch = batch.to(device)
            B = batch.images.size(0)
            total_samples += B

            # 1. Forward Pass
            # Falls Attributionen benötigt werden, brauchen wir Gradients am Input
            with torch.enable_grad():
                batch.images.requires_grad_(True)
                class_logits, concept_logits, features_last_cnn_layer = model(batch.images)

                # 2. Attribution & EPG Loss (falls aktiv)
                if epg_criterion and attributor:
                    attribution_maps = get_attribution_maps(
                        concepts=batch.concepts,
                        features_cnn_layer=features_last_cnn_layer,
                        concept_logits=concept_logits,
                        attributor=attributor,
                        images=batch.images,
                        device=device,
                    )
                    
                    # EPG Loss
                    loss_epg = epg_criterion(attribution_maps, batch.mask_foregrounds, batch.concepts)
                    total_epg_loss += loss_epg.item() * B

                    # Dice & IoU Berechnung (Vergleich Attribution mit Ground Truth Maske)
                    # Hier wird oft ein Threshold auf die Attribution Maps angewendet
                    preds_mask = (attribution_maps > 0.5).float()
                    target_mask = batch.mask_foregrounds.float()
                    
                    intersection = (preds_mask * target_mask).sum(dim=(1, 2))
                    union = (preds_mask + target_mask).clamp(0, 1).sum(dim=(1, 2))
                    
                    dice = (2 * intersection) / (preds_mask.sum(dim=(1, 2)) + target_mask.sum(dim=(1, 2)) + 1e-8)
                    iou = intersection / (union + 1e-8)
                    
                    total_dice += dice.mean().item() * B
                    total_iou += iou.mean().item() * B

            # 3. Klassische Verluste (Concept & Label)
            loss_concepts = concept_criterion(concept_logits, batch.concepts)
            loss_class = classification_criterion(class_logits, batch.labels)
            
            total_concept_loss += loss_concepts.item() * B
            total_class_loss += loss_class.item() * B

            # 4. Konzept-Metriken (Predictions)
            preds_concepts = concept_logits.sigmoid() > 0.5
            targets_concepts = batch.concepts.bool()

            tp_concepts += (preds_concepts & targets_concepts).sum().item()
            fp_concepts += (preds_concepts & ~targets_concepts).sum().item()
            fn_concepts += (~preds_concepts & targets_concepts).sum().item()
            tn_concepts += (~preds_concepts & ~targets_concepts).sum().item()

            all_concept_preds.append(preds_concepts.cpu())
            all_concept_targets.append(targets_concepts.cpu())

            # 5. Label-Metriken (Predictions)
            preds_class = class_logits.argmax(dim=1)
            # Falls Labels One-Hot sind, argmax nutzen, sonst direkt targets_class = batch.labels
            targets_class = batch.labels.argmax(dim=1) if batch.labels.dim() > 1 else batch.labels

            correctly_classified += (preds_class == targets_class).sum().item()
            all_label_preds.extend(preds_class.cpu().numpy())
            all_label_targets.extend(targets_class.cpu().numpy())

    # --- Finale Metrik-Berechnung ---
    
    # Durchschnitte berechnen
    avg_dice = total_dice / total_samples
    avg_iou = total_iou / total_samples
    label_accuracy = correctly_classified / total_samples
    avg_epg_loss = total_epg_loss / total_samples
    
    # Konzept-Genauigkeit (insgesamt über alle Slots)
    acc_concepts = (tp_concepts + tn_concepts) / (tp_concepts + fp_concepts + fn_concepts + tn_concepts + 1e-8)
    
    # Precision / Recall (Micro)
    prec_concepts = tp_concepts / (tp_concepts + fp_concepts + 1e-8)
    rec_concepts = tp_concepts / (tp_concepts + fn_concepts + 1e-8)
    concept_f1_micro = 2 * (prec_concepts * rec_concepts) / (prec_concepts + rec_concepts + 1e-8)

    # F1-Scores (Macro & Weighted)
    all_concept_preds_np = torch.cat(all_concept_preds, dim=0).numpy()
    all_concept_targets_np = torch.cat(all_concept_targets, dim=0).numpy()
    
    concept_f1_macro = f1_score(all_concept_targets_np, all_concept_preds_np, average="macro", zero_division=0)
    label_f1 = f1_score(all_label_targets, all_label_preds, average="weighted", zero_division=0)

    print(
        f"[Eval] Label-Acc: {label_accuracy:.4f} | Label-F1: {label_f1:.4f} | "
        f"Concept-F1 (micro): {concept_f1_micro:.4f} | Dice: {avg_dice:.4f}"
    )

    return EvalMetrics(
        dice_mean=avg_dice,
        iou_mean=avg_iou,
        concept_f1_micro=concept_f1_micro,
        concept_f1_macro=float(concept_f1_macro),
        precision_concepts=prec_concepts,
        recall_concepts=rec_concepts,
        accuracy_concepts=acc_concepts,
        f1_labels=float(label_f1),
        accuracy_labels=label_accuracy,
        avg_epg_loss=avg_epg_loss,
        foreground_dice_scores=avg_dice, # Oder spezifische Scores pro Klasse falls nötig
    )

In [8]:
def calc_test_metrics(data, tags:list[str], n_runs:int, reference_measure:str, test_results_path:str): 
    
    if os.path.exists(test_results_path):
        df_test_results = pd.read_parquet(test_results_path)
    else:
        df_test_results = pd.DataFrame(columns=['run_id'])

    
    summary = create_summary_table(data, filter_tag=tags, reference_measure=reference_measure)

    assert len(summary) == n_runs, f"Erwartet {n_runs} Runs in der Summary Tabelle, aber gefunden: {len(summary)}"

    for _, row in tqdm(summary.iterrows()):
        dataset = row['Dataset'].lower()
        run_id = row['run_id']
        epoch = row['epoch']

        checkpoint_path = f"/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/blobs/checkpoints/cbm_epg_{dataset}_run_{run_id}_epoch{epoch}.pth"

        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(f"Checkpoint {checkpoint_path} existiert nicht.")
        
    print(f"Berechne Test-Metriken für {len(summary)} Runs...")
        

        
    for _, row in summary.iterrows():

        if row['run_id'] in df_test_results['run_id'].values:
            
            print(f"Skip {row['run_id']} (bereits in Test-Ergebnissen vorhanden)")
            continue

        
        train_dataset, val_dataset, test_dataset, n_concepts, n_classes, _ = get_datasets(dataset_name=row['Dataset'], 
                                                            root_dir='/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/datasets', 
                                                            img_size=256, 
                                                            use_soft_labels=False, concept_masks_scale='small',
                                                            attr_level='image')

        _, _, test_loader = get_dataloader(batch_size=4, num_workers=14, 
                                            train_dataset=train_dataset, val_dataset=val_dataset, 
                                            test_dataset=test_dataset)
        
        model = init_from_checkpoint(run_id=row['run_id'], epoch=row['epoch'], dataset=row['Dataset'].lower(), device=device)
        model = model.to(device)

        attributor_mapping = {
            "GradCAM": "GradCamAttributor", 
            "GradCAM++": "GradCamPlusPlusAttributor",
            r"Input $\times$ Gradients": "InputTimesGradientAttributor",
        }

        if isinstance(row['Attribution Method'], str):
            attributor_name = attributor_mapping[row['Attribution Method']]
            attributor = get_attributor_by_name(attributor_name, (256,256))
        else:
            attributor=None
        
        results = evaluate(
            model=model,
            val_loader=test_loader,
            device=str(device),
            concept_criterion=BCEWithLogitsLoss(),
            classification_criterion=CELossFunnyBirds(),
            epg_criterion=EPGLoss(),
            attributor=attributor
        )
        results = results.to_dataframe()
        results['run_id'] = row['run_id']
        results['epoch'] = row['epoch']
        results['dataset'] = row['Dataset']
        df_test_results = pd.concat([df_test_results, results])

        df_test_results.to_parquet(test_results_path)



    attributor_mapping = {
        "GradCamAttributor": "GradCAM",
        "GradCamPlusPlusAttributor": "GradCAM++",
        "InputTimesGradientAttributor": r"Input $\times$ Gradients",
    }
    print(df_test_results.columns)

    return df_test_results

    # df_test_results["Attribution Method"] = df_test_results["Attribution Method"].replace(attributor_mapping)

    # df_test_results["Attribution Method"] = pd.Categorical(
    #     df_test_results["Attribution Method"],
    #     categories=list(attributor_mapping.values()),
    #     ordered=True
    # )

    # # 3️⃣ Sortieren
    # df_sorted = df_test_results.sort_values(
    #     by=["Attribution Method"],
    #     ascending=True
    # )


    # return df_sorted

In [9]:
data.columns

Index(['_step', 'Loss EPG', 'F1 Concepts', 'F1 Labels', 'Runtime', 'Epoch',
       'run_id', 'Attribution Method', '$\lambda_\mathrm{EPG}$', 'EPG-level',
       'config_label', 'Dataset', 'tag_result', 'tag_baseline'],
      dtype='object')

In [10]:
pd.read_parquet('thesis-figures/epg_cbm/results/2_test_metrics_base.parquet')

,run_id,Mean Dice,Mean IoU,Concept Activations $F_1$-Score (Macro),Concept Activations $F_1$-Score (Micro),Precision Concepts,Recall Concepts,Concept Accuracy,EPG Loss,Label $F_1$-Score,Label Accuracy,Foreground Dice,Concept Module,Segmentation Module,Dataset,Runtime,epoch,dataset
0,09st4kbp,0.158710,0.002898,0.479470,0.535727,0.573604,0.502541,0.801947,0.624509,0.696437,0.698999,0.158710,None,None,None,None,35.0,CUB_112
0,0ro02ahu,0.337811,0.042730,0.968602,0.982206,0.982789,0.981624,0.993407,0.375191,0.983032,0.982857,0.337811,None,None,None,None,11.0,FunnyBirds
0,1budbxs6,0.225726,0.047602,0.977086,0.988124,0.989887,0.986366,0.995604,0.623619,0.991463,0.991429,0.225726,None,None,None,None,20.0,FunnyBirds
0,1lmwbjkm,0.275267,0.029500,0.974872,0.982414,0.988010,0.976882,0.993516,0.238994,0.970846,0.971429,0.275267,None,None,None,None,19.0,FunnyBirds
0,1x0ikimu,0.000000,0.000000,0.983756,0.987801,0.991637,0.983995,0.995495,0.000000,0.980442,0.980000,0.000000,None,None,None,None,13.0,FunnyBirds
0,2e4yghk4,0.333076,0.040520,0.965693,0.983500,0.977739,0.989330,0.993846,0.392747,0.988551,0.988571,0.333076,None,None,None,None,4.0,FunnyBirds
0,2v2gu8ot,0.165812,0.003101,0.491876,0.544140,0.576973,0.514842,0.803857,0.622260,0.702438,0.703486,0.165812,None,None,None,None,38.0,CUB_112
0,348pc51f,0.015511,0.000108,0.338115,0.401348,0.407464,0.395413,0.731785,0.570641,0.670641,0.672420,0.015511,None,None,None,None,10.0,CUB_112
0,3gnhxjsw,0.245156,0.023312,0.975411,0.982414,0.988010,0.976882,0.993516,0.581324,0.982868,0.982857,0.245156,None,None,None,None,23.0,FunnyBirds
0,3s84xtwq,0.310588,0.015892,0.484417,0.531933,0.579406,0.491650,0.803263,0.201771,0.670833,0.671557,0.310588,None,None,None,None,65.0,CUB_112


In [ ]:
TEST_RESULTS_PATH_CON = 'thesis-figures/epg_cbm/results/2_test_metrics_base.parquet'

df_test_results = calc_test_metrics(data=data, tags=['result'], n_runs=86, reference_measure=r"F1 Labels", test_results_path=TEST_RESULTS_PATH_CON)


Erstelle Summary Tabelle (Max-Werte)...


86it [00:03, 24.06it/s]


Berechne Test-Metriken für 86 Runs...
Skip 09st4kbp (bereits in Test-Ergebnissen vorhanden)
Skip 0ro02ahu (bereits in Test-Ergebnissen vorhanden)
Skip 1budbxs6 (bereits in Test-Ergebnissen vorhanden)
Skip 1lmwbjkm (bereits in Test-Ergebnissen vorhanden)
Skip 1x0ikimu (bereits in Test-Ergebnissen vorhanden)
Skip 2e4yghk4 (bereits in Test-Ergebnissen vorhanden)
Skip 2v2gu8ot (bereits in Test-Ergebnissen vorhanden)
Skip 348pc51f (bereits in Test-Ergebnissen vorhanden)
Skip 3gnhxjsw (bereits in Test-Ergebnissen vorhanden)
Skip 3s84xtwq (bereits in Test-Ergebnissen vorhanden)
Skip 420g2tyz (bereits in Test-Ergebnissen vorhanden)
Skip 44xnvjn2 (bereits in Test-Ergebnissen vorhanden)
Skip 4bqszj1g (bereits in Test-Ergebnissen vorhanden)
Skip 4stuip2a (bereits in Test-Ergebnissen vorhanden)
Skip 4u2dlwgt (bereits in Test-Ergebnissen vorhanden)
Skip 4z2q89qz (bereits in Test-Ergebnissen vorhanden)
Skip 5ffhnqjw (bereits in Test-Ergebnissen vorhanden)
Skip 5yyk8vr9 (bereits in Test-Ergebnissen v

[Eval]: 100%|██████████| 1449/1449 [09:05<00:00,  2.66it/s]


[Eval] Label-Acc: 0.6643 | Label-F1: 0.6597 | Concept-F1 (micro): 0.4297 | Dice: 0.3945


...

[Eval]: 100%|██████████| 88/88 [00:08<00:00, 10.05it/s]


[Eval] Label-Acc: 0.9886 | Label-F1: 0.9887 | Concept-F1 (micro): 0.9843 | Dice: 0.0069


[Eval]: 100%|██████████| 88/88 [00:03<00:00, 25.00it/s]


[Eval] Label-Acc: 0.9857 | Label-F1: 0.9858 | Concept-F1 (micro): 0.9863 | Dice: 0.2641


[Eval]:  16%|█▌        | 230/1449 [01:26<04:36,  4.40it/s]

In [ ]:
df_test_results = df_test_results.merge(data[['run_id', 'Attribution Method', 'EPG-level', '$\lambda_\mathrm{EPG}$']], how='left', on='run_id')

In [ ]:
df_test_results.columns

In [ ]:
# 3️⃣ Sortieren
df_sorted = df_test_results.sort_values(
    by=["Attribution Method"],
    ascending=True
)
df_sorted.columns

In [ ]:
metrics = [
    "Label $F_1$-Score",
    "Concept Activations $F_1$-Score (Macro)"
]

table = df_sorted.pivot_table(
    index=["Attribution Method", "$\\lambda_\\mathrm{EPG}$"],
    columns="EPG-level",
    values=metrics,
    aggfunc="mean"
).sort_index()

In [ ]:
table

In [ ]:
df_sorted = df_sorted.set_index(["Attribution Method", "EPG-level"]).sort_index()

In [ ]:

df_sorted.columns = pd.MultiIndex.from_tuples([
    ("EPG Score", "Concept"),
    ("EPG Score", "Image"),
    ("F1", "Concepts"),
    ("F1", "Labels")
])

In [ ]:
l = df_sorted.to_latex(index=False, float_format="%.3f", column_format="llccccccc")
print(l)